In [1]:
import os
from dotenv import load_dotenv, find_dotenv

from langchain_openai import ChatOpenAI

In [3]:
load_dotenv(find_dotenv())

True

In [4]:
llm = ChatOpenAI()

In [5]:
llm.invoke("How will the weather be in munich today?")

AIMessage(content="I'm sorry, I am an AI and I do not have real-time information. I recommend checking a weather website or app for the most up-to-date forecast for Munich.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 36, 'prompt_tokens': 17, 'total_tokens': 53, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-BF1xiW3bZW7MTCoklDvh1iqUpku50', 'finish_reason': 'stop', 'logprobs': None}, id='run-fecee0e9-6913-4686-89a2-4e124bbe8639-0', usage_metadata={'input_tokens': 17, 'output_tokens': 36, 'total_tokens': 53, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

## Using tools

In [6]:
from langchain_core.tools import tool

In [7]:
@tool
def get_weather(location: str):
    """Call to get the current weather."""
    if location.lower() in ["munich"]:
        return "It's 15 degrees Celsius and cloudy."
    else:
        return "It's 32 degrees Celsius and sunny."


In [8]:
@tool
def check_seating_availability(location: str, seating_type: str):
    """Call to check seating availability."""
    if location.lower() == "munich" and seating_type.lower() == "outdoor":
        return "Yes, we still have seats available outdoors."
    elif location.lower() == "munich" and seating_type.lower() == "indoor":
        return "Yes, we have indoor seating available."
    else:
        return "Sorry, seating information for this location is unavailable."

In [9]:
tools = [get_weather, check_seating_availability]

In [10]:
llm_with_tools = llm.bind_tools(tools)

In [11]:
result = llm_with_tools.invoke("How will the weather be in munich today?")

In [12]:
result

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_KoRb7dcX7pQ7Yj5jZyHogsWQ', 'function': {'arguments': '{"location":"Munich"}', 'name': 'get_weather'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 84, 'total_tokens': 101, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-BF1znJjXfcywroWlPCLH2a49m3Mnh', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run-50c1765e-de8e-4166-bc00-61565ef2beb3-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Munich'}, 'id': 'call_KoRb7dcX7pQ7Yj5jZyHogsWQ', 'type': 'tool_call'}], usage_metadata={'input_tokens': 84, 'output_tokens': 17, 'total_tokens': 101, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_

In [13]:
result.tool_calls

[{'name': 'get_weather',
  'args': {'location': 'Munich'},
  'id': 'call_KoRb7dcX7pQ7Yj5jZyHogsWQ',
  'type': 'tool_call'}]

In [14]:
result = llm_with_tools.invoke(
    "How will the weather be in munich today? Do you still have seats outdoor available?"
)
result

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_yBx7ltu4rtaL7e4JUiAHbY59', 'function': {'arguments': '{"location": "munich"}', 'name': 'get_weather'}, 'type': 'function'}, {'id': 'call_cXARQeMyRWBDZiBnlTIb68wQ', 'function': {'arguments': '{"location": "munich", "seating_type": "outdoor"}', 'name': 'check_seating_availability'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 56, 'prompt_tokens': 92, 'total_tokens': 148, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-BF204GfQLIuKvghe52R7nhFsXzIQl', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run-d5b844ac-b89a-43c5-a01c-e4e5a22afa9e-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'munich'}, 'id': 'call_yBx7ltu4rt

In [15]:
result.tool_calls

[{'name': 'get_weather',
  'args': {'location': 'munich'},
  'id': 'call_yBx7ltu4rtaL7e4JUiAHbY59',
  'type': 'tool_call'},
 {'name': 'check_seating_availability',
  'args': {'location': 'munich', 'seating_type': 'outdoor'},
  'id': 'call_cXARQeMyRWBDZiBnlTIb68wQ',
  'type': 'tool_call'}]

## Using messages

In [16]:
from langchain_core.messages import HumanMessage, ToolMessage

In [19]:
messages = [
    HumanMessage("""
        How will the weather be in munich today?"
        Do you still have seats outdoor available?
        """
    )
]

In [20]:
llm_output = llm_with_tools.invoke(messages)

In [21]:
messages.append(llm_output)

In [22]:
messages

[HumanMessage(content='\n        How will the weather be in munich today?"\n        Do you still have seats outdoor available?\n        ', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_r4PkTGXjxVV4WTkk6hixJfHj', 'function': {'arguments': '{"location": "munich"}', 'name': 'get_weather'}, 'type': 'function'}, {'id': 'call_2p1nkvFLDdFWP1XH2zEOFz0k', 'function': {'arguments': '{"location": "munich", "seating_type": "outdoor"}', 'name': 'check_seating_availability'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 56, 'prompt_tokens': 96, 'total_tokens': 152, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-BF21c0q7y46h0yGz1O7lyMyJoQiMB', 'fin

In [23]:
tool_mapping = {
    "get_weather": get_weather,
    "check_seating_availability": check_seating_availability,
}

In [24]:
llm_output.tool_calls

[{'name': 'get_weather',
  'args': {'location': 'munich'},
  'id': 'call_r4PkTGXjxVV4WTkk6hixJfHj',
  'type': 'tool_call'},
 {'name': 'check_seating_availability',
  'args': {'location': 'munich', 'seating_type': 'outdoor'},
  'id': 'call_2p1nkvFLDdFWP1XH2zEOFz0k',
  'type': 'tool_call'}]

In [25]:
for tool_call in llm_output.tool_calls:
    tool = tool_mapping[tool_call["name"].lower()]
    tool_output = tool.invoke(tool_call["args"])
    messages.append(ToolMessage(tool_output, tool_call_id=tool_call["id"]))

In [26]:
messages

[HumanMessage(content='\n        How will the weather be in munich today?"\n        Do you still have seats outdoor available?\n        ', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_r4PkTGXjxVV4WTkk6hixJfHj', 'function': {'arguments': '{"location": "munich"}', 'name': 'get_weather'}, 'type': 'function'}, {'id': 'call_2p1nkvFLDdFWP1XH2zEOFz0k', 'function': {'arguments': '{"location": "munich", "seating_type": "outdoor"}', 'name': 'check_seating_availability'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 56, 'prompt_tokens': 96, 'total_tokens': 152, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-BF21c0q7y46h0yGz1O7lyMyJoQiMB', 'fin

In [27]:
llm_with_tools.invoke(messages)

AIMessage(content='The weather in Munich today is 15 degrees Celsius and cloudy. And yes, there are still seats available outdoors.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 181, 'total_tokens': 206, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-BF230TaOupwWekL2JKRa2q9zGE62W', 'finish_reason': 'stop', 'logprobs': None}, id='run-58ce8310-d8ed-47aa-b63f-0144d1acd765-0', usage_metadata={'input_tokens': 181, 'output_tokens': 25, 'total_tokens': 206, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})